# Step 3 — Raw data-quality audit of the anime and user tables

**Purpose.** Step 1 compared the raw data with the author's description. This stage looks inside the columns that the research questions will depend on, and fixes the `null` username problem found in step 1.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1–3 only read the two small tables and take seconds. Part 4 reads two columns of the 80M-row list table and takes a few minutes, and part 7B reads it once more (about 4 minutes); set `RUN_LISTS = False` in the setup cell to skip both.

**Reading the output.** Each part ends with an *Observations* cell marked TODO. Fill it in after the run, writing what you see before interpreting it.

| Part | Question |
|---|---|
| 1 | Which real values does pandas silently turn into `NaN`? |
| 2 | Anime table: placeholders, impossible values, how reliable is `score`, what can a pre-airing model use? |
| 3 | User table: placeholders, impossible ages and dates, suspicious aggregates |
| 4 | Do the per-user counts in `UserList` agree with the rows in `UserAnimeList`? |
| 5 | The users with no row in the list table: do they have other values, which would mean the list table is missing them? |
| 6 | Clarity checks: numbers that the first run did not print, and hypotheses that were never tested |
| 7 | Checks that parts 1 to 6 did not make: the list table, numeric columns, usernames, duplicates and contradictions inside a row |

**Where this step sits.** Step 3 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). This notebook only **finds and counts** problems. It does not clean anything and it does not compare columns statistically. Cleaning rules are decided in step 4, from the evidence found here.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 4, which reads 80M rows

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

## 1. Real values that pandas turns into `NaN`

By default pandas reads strings such as `null`, `NA` or `None` as missing. In step 1 this made the username `null` disappear. Here every cell of the two small tables is read as raw text, and the cells that pandas would have turned into `NaN` are counted per column. Both tables are then loaded again with only the empty string treated as missing.

### 1a. First look at the values, before counting the hidden missing words (NEW, added after the first run)

**Why this comes first.** Below, we count the words that pandas hides (`null`, `N/A`, `None`, ...) with a list of words that we wrote ourselves. In part 2 we also count placeholders with a list we wrote (`Unknown`, `Not available`, `None`, `?`, `0000-00-00`). Both lists come from what we happened to see, so they can miss other words, and they only find a cell that **equals** one of the words. A cell such as `Fridays at Unknown` is not found. So before any counting, we let the data show us which placeholder-like values exist.

**What the cells do.** The first one loads the two small tables with the safe load. For every text column of the anime table and the user table it counts three things, and prints the values behind each count:
- **Exact candidates:** a cell that is only a placeholder-like word, a symbol or a zero (`?`, `-`, `0`, `none`, `null`, `n/a`, `unknown`, `tba`, `tbd`, `undefined`, `not available`, `0000-00-00`, spaces only), in any capital letters.
- **Partial candidates:** a longer text that contains `unknown`, `not available`, `n/a`, `tba` or `tbd` (for example `Fridays at Unknown`).
- **Very short texts:** two characters or fewer, which are often a typo or a symbol.

The second cell prints the five most frequent values of every text column, so that we can read them with our own eyes: a placeholder is usually one of the most frequent values of its column.

**Limits.** This still looks for patterns that we think of. A placeholder written in an unexpected way (`idk`, `nothing`, `Earth` in `location`) can only be found by reading the most frequent values. The list table (80 million rows) is not scanned here; its text columns are `my_tags` (free text) and the two dates, which part 3 of notebook 02 already inspected.

**Read every hit before calling it a placeholder.** The scan only finds candidates. For example, the username `null` is a real user, so it will appear as an exact candidate in `username` although it is not a placeholder.

**What we do with the result.** If the output shows words that are not in our lists, we add them to the lists below and in part 2 before counting.


In [ ]:
# NEW: placeholder-like cells in the text columns of the anime and user tables
import re

na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)

CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
PARTIAL = r"unknown|not available|n/a|tba|tbd"

def scan_text(df, label):
    print("=====", label)
    rows, details = [], {}
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        exact = s[s.str.match(CANDIDATE, flags=re.IGNORECASE)]
        partial = s[s.str.contains(PARTIAL, flags=re.IGNORECASE, regex=True)]
        partial = partial[~partial.index.isin(exact.index)]
        short = s[(s.str.len() <= 2) & ~s.index.isin(exact.index)]
        rows.append({"column": col, "non_missing": len(s), "exact": len(exact), "partial": len(partial), "short": len(short)})
        details[col] = (exact, partial, short)
    table = pd.DataFrame(rows).set_index("column")
    hits = table[(table[["exact", "partial", "short"]] > 0).any(axis=1)]
    print(hits.sort_values("exact", ascending=False).to_string())
    for col in hits.index:
        exact, partial, short = details[col]
        print(f"\n{col}")
        if len(exact): print("  exact  :", exact.value_counts().head(6).to_dict())
        if len(partial): print("  partial:", partial.value_counts().head(6).to_dict())
        if len(short): print("  short  :", short.value_counts().head(6).to_dict())

scan_text(anime, "anime table")
print()
scan_text(users, "user table")


In [ ]:
# NEW: the five most frequent values of every text column (read them: placeholders are often among them)
def top_values(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        counts = df[col].value_counts().head(n)
        print(f"{col}: " + " | ".join(f"{str(v)[:30]!r} {c:,}" for v, c in counts.items()))

top_values(anime, "anime table")
print()
top_values(users, "user table")


### Observations for the first look

**TODO (fill in after the run):**
- Which columns hold exact candidates, partial candidates or very short texts, and how many? Is any word in the output missing from the lists of part 1 and part 2?
- For `broadcast`: how many cells contain `Unknown` inside a longer text, in addition to the exact ones?
- In the five most frequent values of each column, which ones look like a placeholder (for example in `location`, `title_english`, `studio`, `producer`)?


### 1b. Four checks on the guesses of part 1a (NEW, added after the second run)

**Why.** The first look (1a) produced four guesses that we wrote down as hypotheses. Each one can be turned into a number with the two small tables that are already loaded, so we test them here before we rely on them.

| # | Guess from 1a | What the cell counts | What the result would mean |
|---|---|---|---|
| 1 | Some titles still contain HTML codes such as `&#039;` | The cells with a code like `&#039;` or `&amp;`, per text column, and the most frequent codes | If many cells are affected, titles must be decoded (`html.unescape`) before matching or counting words |
| 2 | The 11th may be an automatic day in start dates | The number of start dates (`aired`) for each day of the month, 1 to 31 | If day 11 is far above the other days, it is a default day and those dates are only month-accurate |
| 3 | January 1 may be a default birth day | The share of January 1 among all birth dates, the most frequent month-day values, and the share of January 1 for some birth years | If January 1 is far above 1/365 (0.27%), only the year of those users can be trusted |
| 4 | `aired` is empty when the date is only partial, and `aired_string` still holds it | For the anime with an empty `aired`: what kind of text `aired_string` holds (full date, month and year, year only, empty). The same for the anime with status `Not yet aired` and an empty `aired` (the 147 of notebook 02) | If `aired_string` holds a year or a month, the release year must be read from `aired_string`, and "not yet aired without a start date" does not mean "without any date" |

**Limits.** The cell only describes what is in the two small tables. It cannot tell us why a default day exists, only that the counts look like one.


In [ ]:
# NEW: four checks on the guesses of part 1a (they only read the two small tables)
import ast, html

# 1. HTML codes in the text columns
ENTITY = r"&#?\w+;"
print("1. Cells with an HTML code (for example &#039; or &amp;)")
for label, df in (("anime", anime), ("users", users)):
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        hit = s[s.str.contains(ENTITY, regex=True)]
        if len(hit):
            codes = hit.str.findall(ENTITY).explode().value_counts().head(5).to_dict()
            print(f"   {label}.{col}: {len(hit):,} of {len(s):,} cells ({len(hit) / len(s):.2%}) | codes: {codes}")
            print(f"      example: {hit.iloc[0][:60]!r} -> {html.unescape(hit.iloc[0])[:60]!r}")

# 2. Day of the month of the start date in `aired`
def aired_from(t):
    try:
        return pd.to_datetime(ast.literal_eval(t).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

start = pd.to_datetime(anime["aired"].map(aired_from), utc=True)
day = start.dt.day.dropna().astype(int)
per_day = day.value_counts().reindex(range(1, 32), fill_value=0)
print()
print(f"2. Day of the month of the start date: {len(day):,} start dates, mean per day {len(day) / 31:.0f}, median day count {per_day.median():.0f}")
print("   count per day:", per_day.to_dict())
print("   top 3 days:", per_day.sort_values(ascending=False).head(3).to_dict())
d11 = start[start.dt.day == 11]
print("   start dates on the 11th, most frequent year-months:", d11.dt.strftime("%Y-%m").value_counts().head(5).to_dict())

# 3. January 1 among the birth dates
bd = pd.to_datetime(users["birth_date"], errors="coerce").dropna()
month_day = bd.dt.strftime("%m-%d")
share_jan1 = (month_day == "01-01").mean()
print()
print(f"3. Birth dates: {len(bd):,} parsed. January 1: {(month_day == '01-01').sum():,} users = {share_jan1:.2%} (uniform days would give {1 / 365:.2%})")
print("   most frequent month-day values:", month_day.value_counts().head(5).to_dict())
by_year = bd[bd.dt.year.between(1985, 2000)]
jan1_by_year = (by_year.dt.strftime("%m-%d") == "01-01").groupby(by_year.dt.year).mean().round(3)
print("   share of January 1 per birth year, 1985-2000:", jan1_by_year.to_dict())

# 4. `aired` against `aired_string`
def kind(t):
    if pd.isna(t) or str(t).strip().lower() in ("", "not available"): return "empty"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", t): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", t): return "month and year"
    if re.search(r"\b\d{4}\b", t): return "year only"
    return "other"

string_kind = anime["aired_string"].map(kind)
no_start = start.isna()
print()
print(f"4. `aired` has no start date for {int(no_start.sum()):,} anime. What `aired_string` holds for them:")
print("  ", string_kind[no_start].value_counts().to_dict())
print("   other texts (first 8):", anime.loc[no_start & (string_kind == "other"), "aired_string"].value_counts().head(8).to_dict())
print("   `aired` has a start date but `aired_string` is empty:", int((~no_start & (string_kind == "empty")).sum()))
unaired = no_start & (anime["status"] == "Not yet aired")
print(f"   status `Not yet aired` and no start date in `aired`: {int(unaired.sum()):,}. What `aired_string` holds:")
print("  ", string_kind[unaired].value_counts().to_dict())
print("   most frequent texts:", anime.loc[unaired, "aired_string"].value_counts().head(6).to_dict())


### Observations for the four checks

**TODO (fill in after the run):**
- Check 1: how many cells have an HTML code, in which columns, and which codes are the most frequent?
- Check 2: is day 11 clearly above the other days? Are the dates on the 11th concentrated in a few year-months?
- Check 3: what is the share of January 1 (compare with 0.27%)? Is it the same for every birth year?
- Check 4: for the anime without an `aired` start date, how many have a full date, a month and year, a year only, or nothing in `aired_string`? What does this say about the 147 `Not yet aired` anime of notebook 02?
- Update the rows 20 to 22 of the summary of step 3 with the counts.


In [ ]:
NA_TOKENS = {"null", "NULL", "NA", "N/A", "n/a", "NaN", "nan", "None", "#N/A", "#NA",
             "-NaN", "-nan", "<NA>", "1.#IND", "1.#QNAN", "-1.#IND", "-1.#QNAN", "#N/A N/A"}

for file_name in ["AnimeList.csv", "UserList.csv"]:
    text = pd.read_csv(paths[file_name], dtype=str, keep_default_na=False)
    hits = {col: int(text[col].isin(NA_TOKENS).sum()) for col in text.columns}
    hits = {col: count for col, count in hits.items() if count}
    print(file_name, "-> columns with hidden NA tokens:", hits or "none")
    for col in hits:
        print("   ", col, text.loc[text[col].isin(NA_TOKENS), col].value_counts().head(5).to_dict())
    del text

# Safe load: only the empty string is missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("\nanime:", anime.shape, "| users:", users.shape)
print("username NaN:", users.username.isna().sum(),
      "| username unique:", users.username.is_unique,
      "| distinct usernames:", f"{users.username.nunique():,}", "(description: 302,675)")
assert users.username.is_unique

### Observations for part 1

**TODO (fill in after the run):** which columns lost real values, how many, and which values. Does `users` now have 302,675 distinct usernames?

## 2. Anime table

Columns that matter for the candidate questions: `type`, `source`, `episodes`, `duration`, `aired_string`, `genre`, `status`, `rating` (age rating), `related` (sequels, prequels), and the audience columns `score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`.

For a model that predicts the score **before an anime airs**, the audience columns (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) are only known after airing, so they must be excluded as leakage.

In [ ]:
def missing_overview(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }).sort_values("missing_pct", ascending=False)


print(anime.dtypes.value_counts().to_dict())
print(missing_overview(anime).head(12))

# Placeholder values that are not NaN but also carry no information
PLACEHOLDERS = {"", "Unknown", "Not available", "None", "?", "0000-00-00"}
text_cols = anime.select_dtypes(exclude="number").columns
placeholder_counts = pd.Series({c: int(anime[c].isin(PLACEHOLDERS).sum()) for c in text_cols})
print("\nplaceholder values per column:")
print(placeholder_counts[placeholder_counts > 0].sort_values(ascending=False))

print("\nduplicated anime_id:", anime.anime_id.duplicated().sum())

In [ ]:
# Numeric columns: are the ranges plausible?
num_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes"]
print(anime[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

# How reliable is the published score?
print("\nscore == 0:", (anime.score == 0).sum(),
      "| scored_by == 0:", (anime.scored_by == 0).sum(),
      "| both:", ((anime.score == 0) & (anime.scored_by == 0)).sum())
for k in (10, 50, 100, 1000):
    share = (anime.scored_by < k).mean()
    print(f"anime with scored_by < {k:>4}: {(anime.scored_by < k).sum():>6,} ({share:.1%})")
print("episodes == 0:", (anime.episodes == 0).sum())

In [ ]:
# Categorical columns
for col in ["type", "source", "status", "rating"]:
    print(f"\n{col}: {anime[col].nunique(dropna=False)} distinct values")
    print(anime[col].value_counts(dropna=False).head(12))

In [ ]:
# Release year from aired_string (the first 4-digit number is the start year)
aired_year = pd.to_numeric(anime["aired_string"].str.extract(r"(\d{4})", expand=False), errors="coerce")
print("aired_string without a year:", aired_year.isna().sum(), f"({aired_year.isna().mean():.1%})")
print(aired_year.describe().round(1))
print(anime.loc[aired_year.isna(), "aired_string"].value_counts().head(5))
print("examples:", anime["aired_string"].dropna().head(5).tolist())

# Episode length in minutes from the text in `duration`
print("\n", anime["duration"].value_counts().head(8))
hours = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*hr")[0], errors="coerce").fillna(0)
minutes = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*min")[0], errors="coerce").fillna(0)
has_time = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
duration_min = (hours * 60 + minutes).where(has_time)
print("duration not parsable:", (~has_time).sum(), f"({(~has_time).mean():.1%})")
print(duration_min.describe(percentiles=[.01, .5, .99]).round(1))

In [ ]:
# genre holds several values in one cell
genres = anime["genre"].dropna().str.split(", ")
print("anime without genre:", anime["genre"].isna().sum())
print("genres per anime:", genres.str.len().describe().round(2).to_dict())
print("distinct genres:", genres.explode().nunique())
print(genres.explode().value_counts().head(10))

# related: how are sequels stored, and how many anime have one?
print("\nexample:", anime["related"].dropna().head(2).tolist())
for relation in ["Sequel", "Prequel", "Side story", "Spin-off"]:
    share = anime["related"].str.contains(f"'{relation}'", na=False).mean()
    print(f"anime with a '{relation}' entry: {share:.1%}")

### Observations for part 2

**TODO (fill in after the run):**
- Placeholders and missingness: which columns are unusable or need a placeholder-to-`NaN` step?
- Implausible values: `episodes == 0`, extreme `duration`, release years outside 1900–2018.
- Reliability of `score`: how many anime have very few raters?
- Whether `related` is parsable enough for the sequel question, and whether `aired_string` and `duration` can be parsed for the time-trend and ML questions.

## 3. User table

In [ ]:
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
stat_cols = ["user_days_spent_watching", "stats_mean_score", "stats_rewatched", "stats_episodes"]

print(users.dtypes.value_counts().to_dict())
print(missing_overview(users).head(8))
print("access_rank non-missing:", users["access_rank"].notna().sum())

print("\ngender:")
print(users["gender"].value_counts(dropna=False))
print("\ndistinct locations:", users["location"].nunique(),
      "| missing:", users["location"].isna().sum())
print(users["location"].value_counts().head(10))

In [ ]:
# Dates: placeholders and implausible values
for col in ["birth_date", "join_date", "last_online"]:
    parsed = pd.to_datetime(users[col], errors="coerce")
    print(f"{col:12s} unparsable/missing: {parsed.isna().sum():>7,}",
          "| '0000-00-00':", int((users[col] == "0000-00-00").sum()),
          "| earliest:", parsed.min(), "| latest:", parsed.max())
    users[col + "_dt"] = parsed

age_2018 = (pd.Timestamp("2018-01-01") - users["birth_date_dt"]).dt.days / 365.25
print("\nage at 2018-01-01:")
print(age_2018.describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
print("age < 8:", int((age_2018 < 8).sum()), "| age > 80:", int((age_2018 > 80).sum()))
print("last_online before 2000:", int((users["last_online_dt"].dt.year < 2000).sum()),
      "| joined after last_online:", int((users["join_date_dt"] > users["last_online_dt"]).sum()))

# Aggregates reported per user
print(users[stat_cols].describe(percentiles=[.01, .5, .99, .999]).T.round(2))
print("stats_mean_score == 0:", int((users["stats_mean_score"] == 0).sum()))
print("users declaring no anime at all:", int((users[count_cols].sum(axis=1) == 0).sum()))

### Observations for part 3

**TODO (fill in after the run):** which gender and location values need grouping or cleaning, how many ages are impossible, whether `last_online` and `join_date` look reliable, and whether any aggregate has extreme values (candidates for truncation or robust statistics).

## 4. Do the per-user counts agree with the list table?

`UserList` reports how many anime each user has in each status. `UserAnimeList` holds the actual rows. If both come from the same crawl, the counts should match. This also settles the 8 users in step 1 who declare anime but have no rows.

In [ ]:
if RUN_LISTS:
    lists = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "my_status"],
        dtype={"username": "category", "my_status": "int8"}, **na_kwargs,
    )
    print(f"list rows: {len(lists):,} | username NaN: {lists.username.isna().sum()}")

    per_status = lists.groupby(["username", "my_status"], observed=True).size().unstack(fill_value=0)
    print("usernames in lists but not in users:", len(set(per_status.index) - set(users.username)))

    status_to_col = {1: "user_watching", 2: "user_completed", 3: "user_onhold",
                     4: "user_dropped", 6: "user_plantowatch"}
    actual = per_status.reindex(columns=list(status_to_col)).fillna(0).astype(int)
    actual.columns = [c + "_actual" for c in status_to_col.values()]
    actual.index = actual.index.astype(object)
    declared = users.set_index("username")[count_cols].add_suffix("_declared")
    merged = declared.join(actual, how="left").fillna(0)
    print("merge rows:", len(merged), "| users:", len(users))

    for col in count_cols:
        diff = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} exact match: {(diff == 0).mean():6.1%} | median diff: {diff.median():.0f}"
              f" | mean abs diff: {diff.abs().mean():.2f}")

    declared_total = merged[[c + "_declared" for c in count_cols]].sum(axis=1)
    actual_total = merged[[c + "_actual" for c in count_cols]].sum(axis=1)
    print("\ndeclare 0 but have rows:", int(((declared_total == 0) & (actual_total > 0)).sum()),
          "| declare > 0 but have no rows:", int(((declared_total > 0) & (actual_total == 0)).sum()))
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 4

**TODO (fill in after the run):** how often do declared and actual counts agree, in which direction do they differ, and how many users declare anime but have no rows? If the counts disagree widely, `UserList` aggregates should not be used as ground truth for the questions.

## 5. The users with no row in the list table: is the list table missing them?

In step 1 we found **19,630 users** who exist in `UserList` but own no row in `UserAnimeList`. For 19,622 of them the profile declares 0 anime, so an empty list is the natural explanation. But a declared count is only one column. If these users also have **other activity values** (days spent watching, episodes watched, a mean score), their list may exist on MyAnimeList and simply be missing from our list table.

| What we see among these users | What it means |
|---|---|
| `user_days_spent_watching`, `stats_episodes` and `stats_mean_score` are all 0 | The list is really empty. Nothing is missing |
| Many have positive values there | They watched and rated anime, so their list is **missing from the list table** (not crawled, private, or lost) |
| Their profile fields (gender, birth date, location) are filled as often as for other users | They are normal active profiles. If those fields are mostly empty, they are probably inactive accounts |

In [ ]:
if RUN_LISTS:
    users["has_rows"] = users["username"].isin(actual.index)
    no_rows = users[~users["has_rows"]]
    with_rows = users[users["has_rows"]]
    declared_size = no_rows[count_cols].sum(axis=1)
    print(f"users without a row in the list table: {len(no_rows):,} | with rows: {len(with_rows):,}")
    print(f"  declared list size 0: {int((declared_size == 0).sum()):,} | declared list size > 0: {int((declared_size > 0).sum()):,}")

    print("\nactivity values of the users without rows (an empty account would show 0 everywhere):")
    for col in ["user_days_spent_watching", "stats_episodes", "stats_rewatched", "stats_mean_score"]:
        s = no_rows[col]
        print(f"  {col:26s} missing: {int(s.isna().sum()):>6,} | zero: {int((s == 0).sum()):>6,} | positive: {int((s > 0).sum()):>6,}"
              f" | max: {s.max():,.1f}")
    active = ((no_rows["user_days_spent_watching"] > 0) | (no_rows["stats_episodes"] > 0) | (no_rows["stats_mean_score"] > 0))
    print(f"\nusers without rows that show activity in at least one of the three columns: {int(active.sum()):,}")

    print("\nprofile fields filled (share of users), compared with the users that have rows:")
    fields = ["gender", "birth_date", "location", "join_date", "last_online"]
    print(pd.DataFrame({"no_rows": no_rows[fields].notna().mean(),
                        "with_rows": with_rows[fields].notna().mean()}).round(3).to_string())

    print("\nusers without rows that declare anime (the unexplained group):")
    cols_show = ["username"] + count_cols + ["user_days_spent_watching", "stats_episodes", "stats_mean_score"]
    print(no_rows.loc[declared_size > 0, cols_show].sort_values("user_completed", ascending=False).head(15).to_string(index=False))
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 5

**TODO (fill in after the run):** How many of the 19,630 users show activity (days, episodes or a mean score above 0)? Are the 19,622 users with a declared size of 0 also 0 in the three activity columns? Are their profile fields filled as often as for the users with rows? What do the 8 users who declare anime look like? Decide: are these users empty accounts (drop them from list analyses) or users whose list is missing (report how many)?

## 6. Clarity checks (NEW, added after the first run)

Parts 1 to 5 ran once. Some sentences of their observations rest on numbers that the first run did not print, and some hypotheses were never tested. This part prints them. It changes nothing in the data, and it reuses the objects built above (`anime`, `users`, `age_2018`, `merged`, ...), so run the notebook from the top.

| Check | What it settles |
|---|---|
| A. Anime values | How many `rank` and `popularity` are 0? Which words fill `broadcast`? What are the 313 `duration` texts that cannot be parsed? Which anime have `score = 0` but raters? |
| B. User dates and ages | Is `last_online` before 2000 always the year 1900 (our hypothesis)? Which `birth_date` texts cannot be parsed? Which birth years give the impossible ages? |
| C. Extremes | How many users lie above several caps of `stats_episodes` and `user_days_spent_watching`, and what do the largest ones look like? This helps to choose a cap in step 4. |
| D. `deactivated` usernames | How many usernames start with `deactivated`, and do they own list rows? (One of the 8 users who declare anime without rows is `deactivated-180120`.) |
| E. Declared against actual counts | In which direction do the counts differ, and who are the 6 users that declare 0 anime but have rows? |


In [ ]:
# NEW A: anime values that the first run did not print
print("rank == 0:", int((anime["rank"] == 0).sum()), "| rank missing:", int(anime["rank"].isna().sum()),
      "| popularity == 0:", int((anime["popularity"] == 0).sum()))

print("\nbroadcast: the words that count as placeholders")
print(anime.loc[anime["broadcast"].isin(PLACEHOLDERS), "broadcast"].value_counts().to_string())

has_time_text = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
print("\nduration: the texts that cannot be parsed into minutes")
print(anime.loc[~has_time_text, "duration"].value_counts(dropna=False).head(10).to_string())

odd_score = (anime["score"] == 0) & (anime["scored_by"] > 0)
print("\nscore == 0 although scored_by > 0:", int(odd_score.sum()))
print(anime.loc[odd_score, ["anime_id", "title", "type", "status", "scored_by", "members"]]
      .sort_values("scored_by", ascending=False).head(10).to_string(index=False))


In [ ]:
# NEW B: user dates and ages
last_year = users["last_online_dt"].dt.year
print("last_online before 2000, by year:", last_year[last_year < 2000].value_counts().sort_index().to_dict())
print("last_online year counts from 2000 on:", last_year[last_year >= 2000].value_counts().sort_index().to_dict())

unparsable = users["birth_date"].notna() & users["birth_date_dt"].isna()
print("\nbirth_date texts that cannot be parsed:", users.loc[unparsable, "birth_date"].tolist())

birth_year = users["birth_date_dt"].dt.year
print("birth years of the users under 8 at 2018-01-01:", birth_year[age_2018 < 8].value_counts().head(8).to_dict())
print("birth years of the users over 80 at 2018-01-01:", birth_year[age_2018 > 80].value_counts().head(8).to_dict())


In [ ]:
# NEW C: how many users lie above several caps, and what do the largest look like?
caps = {"stats_episodes": [10_000, 50_000, 100_000, 1_000_000],
        "user_days_spent_watching": [365, 1_000, 3_650, 36_500]}
for col, values in caps.items():
    print(col, {f"> {v:,}": int((users[col] > v).sum()) for v in values})

print()
show = ["username", "stats_episodes", "user_days_spent_watching", "user_completed", "user_plantowatch"]
print(users.nlargest(8, "stats_episodes")[show].to_string(index=False))


In [ ]:
# NEW D: usernames that start with "deactivated"
deactivated = users[users["username"].str.lower().str.startswith("deactivated")]
print("usernames starting with 'deactivated':", len(deactivated))
if "has_rows" in users:
    print("  of them with at least one list row:", int(deactivated["has_rows"].sum()))
print(deactivated[["username", "user_completed", "user_plantowatch", "user_days_spent_watching"]].head(10).to_string(index=False))


In [ ]:
# NEW E: declared against actual counts, direction of the differences
if RUN_LISTS:
    for col in count_cols:
        d = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} actual > declared: {int((d > 0).sum()):>6,} | actual < declared: {int((d < 0).sum()):>6,}"
              f" | largest gap: {int(d.abs().max()):,}")

    zero_declared = merged[(declared_total == 0) & (actual_total > 0)]
    print("\nusers that declare 0 anime but have rows:")
    print(zero_declared[[c + "_actual" for c in count_cols]].assign(total_rows=actual_total[zero_declared.index]).to_string())
else:
    print("Skipped (RUN_LISTS = False)")


### Observations for the clarity checks

**TODO (fill in after the run):**
- A: how many `rank` and `popularity` are 0? Which words fill `broadcast`, and which texts make `duration` unparsable? Are the anime with `score = 0` and raters few or many?
- B: is every `last_online` before 2000 in the year 1900? If yes, the hypothesis that the year was lost is supported. Which birth years give the impossible ages?
- C: at which cap do few enough users lie above it? Are the largest users plausible?
- D: how many usernames start with `deactivated`, and do they own rows?
- E: is the actual count usually larger or smaller than the declared one? Who are the 6 users that declare 0 but have rows?


## 7. Checks that parts 1 to 6 did not make: the list table, numeric columns, usernames, duplicates and contradictions (NEW)

**Purpose.** When we read parts 1 to 6 again as a whole, we found places where we looked at one table or one kind of column and not at the others, or where we described a column without testing it. This part runs those checks, so that the audit has no blind spot before the cleaning decisions of step 4. It changes nothing in the data and reuses `anime`, `users`, `paths` and `RUN_LISTS` from the cells above.

**How to run.** Part 7A only reads the two small tables (seconds). Part 7B reads the 80M-row list table once, 3M rows at a time (about 4 minutes); `RUN_LISTS = False` skips it. The pass and its reports are separate cells, so a mistake in a report does not need a second pass.

**What was missing, and why it matters.**

| # | Gap in notebook 03 | Why it matters | Where |
|---|---|---|---|
| 1 | The list table (80M rows, the largest table) was never scanned for placeholders. The first look (1a) skipped it on purpose because of its size. | A placeholder in the largest table affects the most rows. `my_tags` is free text, and the two date columns can hold default or impossible dates | B1 to B3 |
| 2 | Numeric columns were never scanned for placeholder numbers (`-1`, `999`, `9999`). The first look (1a) read text columns only. | A number such as `-1` or `9999` standing for "unknown" would enter every mean and correlation | A1 |
| 3 | The usernames were checked for being unique (exact match), not for **capital letters and spaces**. | The username is the join key between `UserList` and `UserAnimeList`. Two spellings of one name, or a trailing space, break a join without any error | A2, B4 |
| 4 | `location` was only counted by its most frequent values. | If `USA`, `usa` and `Usa ` are three values, every grouping by country is split | A3 |
| 5 | Duplicates were checked for `anime_id` only. | The same anime may exist under two ids, which would double its weight | A4 |
| 6 | No check that two columns of the same row agree (`scored_by` against `members`, end date against start date, a finished anime that starts after the crawl). | A row that contradicts itself is wrong in at least one column | A5, A5b, A5c, B3 |
| 7 | The columns that hold Python text (`aired`, `related`, `opening_theme`, `ending_theme`) were read with `ast.literal_eval` in notebook 02 without counting the failures. | A parse failure silently drops a row from any analysis of these columns | A6 |
| 8 | The dates of the list table were seen as a top-5 list in notebook 02, and never tested for partial dates, impossible dates, dates after the crawl, or a finish date before the start date. `my_last_updated` was only used for its maximum. | A date range decides which rows can be used for a time analysis | B3 |
| 9 | `anime_id` of the list table was never checked against `AnimeList`, and `my_watched_episodes` was never compared with the episode count. | An id without an anime cannot be joined. More episodes watched than exist is impossible | B4, B5 |

**Evidence tags.** Everything printed is a count from the files ([V]). Explanations of *why* a value exists are hypotheses and are labelled so.


In [ ]:
# Setup for part 7 (the objects `anime`, `users`, `paths`, `na_kwargs` and `RUN_LISTS` come from the cells above)
from collections import Counter
import ast, html, re

CRAWL_END = pd.Timestamp("2018-05-25", tz="UTC")  # estimated end of the crawl (notebook 02)


### 7A. The two small tables

### A1. Numeric columns: impossible values and placeholder numbers

For every numeric column (the id columns are skipped) the cell prints how many cells are missing, the minimum, the maximum, how many values are negative, the five most frequent values, and how many cells equal a number that is often used as a placeholder (`-1`, `99`, `999`, `9999`, `99999`). A value such as `999` can also be real (for example a count of favorites), so a hit is a reason to look, not a proof.


In [ ]:
# A1. Numeric columns: impossible values, most frequent values, placeholder-like numbers
SENTINELS = [-1, 99, 999, 9999, 99999]

def numeric_overview(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(include="number").columns:
        if col.endswith("_id"):
            continue
        s = df[col].dropna()
        top = " | ".join(f"{v:,.6g} x{c:,}" for v, c in s.value_counts().head(n).items())
        sentinel = {v: int((s == v).sum()) for v in SENTINELS if (s == v).any()}
        extra = f" | placeholder-like: {sentinel}" if sentinel else ""
        print(f"{col}: missing {int(df[col].isna().sum()):,} | min {s.min():,.6g} | max {s.max():,.6g}"
              f" | negative {int((s < 0).sum()):,} | top: {top}{extra}")

numeric_overview(anime, "anime table")
print()
numeric_overview(users, "user table")


### A2. Spaces, capital letters and the username (the join key)

The first cell counts, in every text column of both tables, the cells with a space at the start or at the end, and the cells with two or more spaces in a row. The second one tests the usernames: are there names that differ only by capital letters or by a space? On MyAnimeList a profile address does not depend on capital letters, so two such names could be one person; and a join on the exact text would then miss it.


In [ ]:
# A2. Spaces in text cells, and usernames that differ only by capital letters
def spacing(df, label):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        edge = int((s != s.str.strip()).sum())
        double = int(s.str.contains(r"\s{2,}", regex=True).sum())
        if edge or double:
            print(f"{col}: space at the start or the end {edge:,} | two or more spaces in a row {double:,}")

spacing(anime, "anime table")
print()
spacing(users, "user table")

name = users["username"]
low = name.str.lower()
clash = low.duplicated(keep=False)
print()
print("usernames:", f"{len(name):,}", "| distinct:", f"{name.nunique():,}", "| distinct after lowercase:", f"{low.nunique():,}")
print("names that differ only by capital letters:", int(clash.sum()), "in", low[clash].nunique(), "groups")
print(users.loc[clash, ["username", "user_id"]].sort_values("username", key=lambda s: s.str.lower()).head(10).to_string(index=False))
print("usernames with a space at the start or the end:", int((name != name.str.strip()).sum()))


### A3. `location`: how many spellings hide behind the same place?

`location` is free text. The cell compares the number of distinct raw texts with the number after we remove capital letters and extra spaces, and prints the places that have the most spellings. This tells how much grouping work step 4 has before any analysis by country or city.


In [ ]:
# A3. location: raw texts against texts without capital letters and extra spaces
loc = users["location"].dropna()
norm = loc.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("filled locations:", f"{len(loc):,}", "| distinct raw texts:", f"{loc.nunique():,}", "| distinct after normalising:", f"{norm.nunique():,}")
print("most frequent after normalising:", norm.value_counts().head(10).to_dict())

spellings = loc.groupby(norm).nunique().sort_values(ascending=False)
print("\nplaces written in the most different ways:")
for key in spellings.head(5).index:
    forms = loc[norm == key].value_counts().head(6).to_dict()
    print(f"  {key!r}: {int(spellings[key])} spellings, for example {forms}")


### A4. Duplicates beyond `anime_id`

Notebook 03 only checked that `anime_id` is unique. Here we look for the same title under several ids, and for rows that are identical in every column except `anime_id` (the same anime entered twice).


In [ ]:
# A4. Duplicate anime that have different ids
dup_title = anime[anime["title"].duplicated(keep=False)]
print("titles used by more than one anime_id:", dup_title["title"].nunique(), "| rows:", len(dup_title))
print(dup_title.sort_values("title")[["anime_id", "title", "type", "status", "scored_by"]].head(10).to_string(index=False))

same_but_id = anime.drop(columns=["anime_id"]).duplicated(keep=False)
print("\nrows identical in every column except anime_id:", int(same_but_id.sum()))

title_clean = anime["title"].str.strip().str.lower()
print("titles that differ only by capital letters or spaces:", int(title_clean.duplicated(keep=False).sum() - anime["title"].duplicated(keep=False).sum()))


### A5. Do the columns of one row agree with each other?

A row that contradicts itself is wrong in at least one column. The cell counts: a `score` outside 0 to 10; more raters than members (`scored_by` > `members`); more favorites than members; an end date before the start date in `aired`; a **finished** anime whose start date lies after the crawl; and a **currently airing** anime whose end date lies before the crawl (the crawl ended about 2018-05-25, see notebook 02).


In [ ]:
# A5. Contradictions inside one row of the anime table
def aired_part(t, key):
    try:
        return pd.to_datetime(ast.literal_eval(t).get(key), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

air_start = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "from")), utc=True)
air_end = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "to")), utc=True)

print("score outside 0 to 10:", int(((anime["score"] < 0) | (anime["score"] > 10)).sum()))
print("scored_by > members:", int((anime["scored_by"] > anime["members"]).sum()),
      "| favorites > members:", int((anime["favorites"] > anime["members"]).sum()))
print("end date before start date:", int((air_end < air_start).sum()))
finished_late = (anime["status"] == "Finished Airing") & (air_start > CRAWL_END)
airing_over = (anime["status"] == "Currently Airing") & (air_end < CRAWL_END)
print("Finished Airing with a start date after the crawl:", int(finished_late.sum()))
print("Currently Airing with an end date before the crawl:", int(airing_over.sum()))
cols = ["anime_id", "title", "status", "aired_string"]
print(anime.loc[finished_late, cols].head(5).to_string(index=False))
print(anime.loc[airing_over, cols].head(5).to_string(index=False))


### A5b. The raw `aired` text of the rows that contradict themselves (NEW)

A5 found rows that contradict themselves, but it printed only the `aired_string`. For the first five "Finished Airing" anime, `aired_string` shows an old date (for example `Oct, 2007 to Oct, 2008`), yet A5 says their start date in `aired` lies after the crawl. So `aired` and `aired_string` seem to disagree for them. This cell prints the raw `aired` text of every group, and tests one guess: **when the end of `aired_string` is only a year or a month, `aired` stores the first day of that period**, which would put the end before the start (for example `Apr 4, 2018 to 2018` would end on 2018-01-01).


In [ ]:
# A5b. The raw `aired` text of the anime that contradict themselves, and the guess about partial end dates
def end_kind(t):
    parts = str(t).split(" to ")
    if len(parts) < 2: return "no end text"
    e = parts[1].strip()
    if e == "?": return "still open"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", e): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

groups = {
    "end date before start date": air_end < air_start,
    "Finished Airing, start date after the crawl": finished_late,
    "Currently Airing, end date before the crawl": airing_over,
    "favorites > members": anime["favorites"] > anime["members"],
}
for label, mask in groups.items():
    print(f"--- {label}: {int(mask.sum())} anime")
    print(anime.loc[mask, ["anime_id", "title", "status", "aired_string", "aired"]].head(40).to_string(index=False, max_colwidth=75))

has_end = air_end.notna()
kinds = anime["aired_string"].map(end_kind)
before_start = groups["end date before start date"]
print()
print("how the end of `aired_string` is written, for the anime that have an end date in `aired`:")
print("  all:", kinds[has_end].value_counts().to_dict())
print("  end before start:", kinds[before_start].value_counts().to_dict())
print(f"end date on the 1st of a month: all {(air_end[has_end].dt.day == 1).mean():.1%} | end before start {(air_end[before_start].dt.day == 1).mean():.1%}")


### A5c. How many `aired` dates are filled in? (NEW)

A5b showed that a date written only partly in `aired_string` (`Oct, 2007`, `2011`) is stored in `aired` as a complete date, and that the same numbers come back again and again: the year 2018, the month May, and the days 9 to 12. Our hypothesis is that the missing parts were filled with the date of the fetch. This cell tests it on **every** anime, not only on the 54 rows that A5b printed.

For the start and for the end of `aired_string` it says how the text is written (a full date, a month and a year, a year only, `?` for an open end, or absent). For each kind it counts the anime that have a date in `aired` and prints the most frequent days, months and years of these dates. For the full dates it also counts how often the date in `aired` differs from the date written in the text (the year shifted by one, for example).


In [ ]:
# A5c. Are the missing parts of a date in aired_string filled in `aired`? (all anime)
def split_aired(t):
    p = str(t).split(" to ")
    return p[0].strip(), (p[1].strip() if len(p) > 1 else None)

def text_kind(e):
    if pd.isna(e): return "absent"
    if e == "?": return "open"
    if re.search(r"^[A-Z][a-z]{2} \d{1,2}, \d{4}$", e): return "full date"
    if re.search(r"^[A-Z][a-z]{2}, \d{4}$", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

pairs = anime["aired_string"].map(split_aired)
for label, pos, stamp in (("start", 0, air_start), ("end", 1, air_end)):
    texts = pairs.map(lambda p: p[pos])
    text_kinds = texts.map(text_kind)
    print(f"--- {label}: how `aired_string` writes it, and what `aired` holds")
    for k in ["full date", "month and year", "year only", "open", "absent", "other"]:
        in_kind = text_kinds == k
        has_date = in_kind & stamp.notna()
        line = f"  {k:15s} anime: {int(in_kind.sum()):>6,} | with a date in aired: {int(has_date.sum()):>6,}"
        if has_date.any():
            s = stamp[has_date]
            line += (f" | day top 4: {s.dt.day.value_counts().head(4).to_dict()}"
                     f" | month top 3: {s.dt.month.value_counts().head(3).to_dict()}"
                     f" | year top 3: {s.dt.year.value_counts().head(3).to_dict()}")
        print(line)
    is_full = text_kinds == "full date"
    written = pd.to_datetime(texts.where(is_full), format="%b %d, %Y", errors="coerce")
    same = stamp.dt.tz_localize(None).dt.normalize() == written
    compared = is_full & stamp.notna() & written.notna()
    print(f"  full dates in the text that also have a date in aired: {int(compared.sum()):,} | aired differs from the text: {int((compared & ~same).sum()):,}")
    print(anime.loc[compared & ~same, ["anime_id", "aired_string", "aired"]].head(8).to_string(index=False, max_colwidth=70))
    print()


### A6. The columns that hold Python text: how many cells cannot be read?

`aired`, `related`, `opening_theme` and `ending_theme` are texts written like a Python dictionary or list. Notebook 02 read them with `ast.literal_eval`, and a cell that fails was silently turned into a missing value. Here the failures are counted.


In [ ]:
# A6. Python-literal columns: how many cells fail to parse?
for col in ["aired", "related", "opening_theme", "ending_theme"]:
    s = anime[col].dropna()
    bad = []
    for v in s:
        try:
            ast.literal_eval(v)
        except Exception:
            bad.append(v)
    example = bad[0][:80] if bad else "-"
    print(f"{col:14s} cells: {len(s):>6,} | cannot be parsed: {len(bad):>5,} | example: {example!r}")


### Observations for part 7A

**TODO (fill in after the run):**
- A1: which numeric columns have negative values, or a placeholder-like number? Do the most frequent values of any column look like a placeholder (for example `0` or `-1`)?
- A2: which text columns have spaces at the start or at the end? How many usernames differ only by capital letters or by a space, and can the pairs be joined to the same person?
- A3: how many raw spellings of `location` collapse after normalising? Which places have the most spellings?
- A4: are there titles shared by several ids, and are some of them true duplicates (rows identical except `anime_id`)?
- A5: how many rows contradict themselves, and which check finds the most?
- A5b: what does the raw `aired` text of the 16 "Finished Airing" anime look like? Does an end given as a year or a month become the first day of that period (compare the share of end dates on the 1st for all anime and for the "end before start" group)?
- A5c: how many start and end dates of `aired` come from a partial text? Which days, months and years do they have? For full dates, how often does `aired` differ from the text?
- A6: does any of the four columns have cells that cannot be parsed?


### 7B. The list table (80M rows), read once

The list table was never scanned for placeholders (the first look of notebook 03 skipped it because of its size). It is read once, 3M rows at a time, and each chunk only updates counters. A text column with a few thousand distinct values (the two dates) is counted exactly; `my_tags`, which has far too many distinct values, is counted by pattern and by an approximate top list (the 300 most frequent values of each chunk are merged, so a frequent value is found and its count is a lower bound).

| Cell | What it does |
|---|---|
| B1 | The one pass. Prints only its progress |
| B2 | Report on `my_tags` |
| B3 | Report on the two date columns and on `my_last_updated` |
| B4 | Report on `anime_id` and on the usernames of the list table |
| B5 | Report on `my_watched_episodes` against the episode count |


In [ ]:
# B1. One pass over the list table: only counters are updated, the reports are in the next cells
if RUN_LISTS:
    CHUNK = 3_000_000
    DATE_OK = r"^\d{4}-\d{2}-\d{2}$"
    REAL_DATE = r"^(?!0000)\d{4}-(?!00)\d{2}-(?!00)\d{2}$"
    CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
    EPOCH_SITE = pd.Timestamp("2004-11-01", tz="UTC").timestamp()   # MyAnimeList did not exist before
    EPOCH_END = CRAWL_END.timestamp() + 86400                       # end of 2018-05-25

    use = ["username", "anime_id", "my_watched_episodes", "my_status", "my_start_date",
           "my_finish_date", "my_last_updated", "my_tags"]
    dtypes = {"anime_id": "int32", "my_watched_episodes": "float32", "my_status": "int8", "my_last_updated": "float64"}
    episodes_of = anime.set_index("anime_id")["episodes"]
    known_ids = set(anime["anime_id"])

    rows = empty_username = orphan_rows = 0
    names, ids = set(), set()
    date_counts = {"my_start_date": Counter(), "my_finish_date": Counter()}
    both_real = finish_before_start = 0
    tag_filled = tag_html = 0
    tag_placeholder, tag_top = Counter(), Counter()
    upd_min, upd_max = float("inf"), float("-inf")
    upd_nonpositive = upd_before_site = upd_after_crawl = 0
    ep_known = watched_over = completed_rows = completed_zero = completed_less = 0

    reader = pd.read_csv(paths["UserAnimeList.csv"], usecols=use, dtype=dtypes, chunksize=CHUNK, **na_kwargs)
    for k, chunk in enumerate(reader, start=1):
        rows += len(chunk)
        empty_username += int(chunk["username"].isna().sum())
        names.update(chunk["username"].dropna().unique().tolist())
        ids.update(chunk["anime_id"].unique().tolist())
        orphan_rows += int((~chunk["anime_id"].isin(known_ids)).sum())

        for col in date_counts:
            date_counts[col].update(chunk[col].fillna("<empty>").value_counts().to_dict())
        st = chunk["my_start_date"].fillna("")
        fi = chunk["my_finish_date"].fillna("")
        real = st.str.match(REAL_DATE) & fi.str.match(REAL_DATE)
        both_real += int(real.sum())
        finish_before_start += int((real & (fi < st)).sum())

        tags = chunk["my_tags"].dropna()
        tag_filled += len(tags)
        if len(tags):
            tag_html += int(tags.str.contains(r"&#?\w+;", regex=True).sum())
            is_placeholder = tags.str.match(CANDIDATE, flags=re.IGNORECASE)
            tag_placeholder.update(tags[is_placeholder].str.lower().value_counts().to_dict())
            tag_top.update(tags.value_counts().head(300).to_dict())

        upd = chunk["my_last_updated"].dropna()
        if len(upd):
            upd_min, upd_max = min(upd_min, float(upd.min())), max(upd_max, float(upd.max()))
            upd_nonpositive += int((upd <= 0).sum())
            upd_before_site += int((upd < EPOCH_SITE).sum())
            upd_after_crawl += int((upd > EPOCH_END).sum())

        watched = chunk["my_watched_episodes"]
        total_eps = chunk["anime_id"].map(episodes_of)
        has_eps = total_eps > 0
        done = chunk["my_status"] == 2
        ep_known += int(has_eps.sum())
        watched_over += int((has_eps & (watched > total_eps)).sum())
        completed_rows += int(done.sum())
        completed_zero += int((done & (watched == 0)).sum())
        completed_less += int((done & has_eps & (watched < total_eps)).sum())

        if k % 5 == 0:
            print(f"  ... {k} chunks, {rows:,} rows")
    print(f"done: {rows:,} rows in {k} chunks")
else:
    print("Skipped (RUN_LISTS = False)")


In [ ]:
# B2. my_tags: how many are filled, which are placeholder-like, which are the most frequent
if RUN_LISTS:
    print(f"my_tags filled: {tag_filled:,} of {rows:,} rows ({tag_filled / rows:.1%}) | with an HTML code: {tag_html:,}")
    print("placeholder-like tags:", f"{sum(tag_placeholder.values()):,}", dict(tag_placeholder.most_common(8)))
    print("most frequent tags (approximate, a lower bound):")
    for v, c in tag_top.most_common(10):
        print(f"   {v[:40]!r} {c:,}")
else:
    print("Skipped (RUN_LISTS = False)")


In [ ]:
# B3. The two date columns and my_last_updated
if RUN_LISTS:
    def summarize_dates(counter, label):
        s = pd.Series(counter, dtype="int64")
        total = int(s.sum())
        txt = pd.Series(s.index, index=s.index)
        is_format = txt.str.match(DATE_OK)
        is_empty = txt == "<empty>"
        is_zero = txt == "0000-00-00"
        other = s[~is_format & ~is_empty]
        real_format = is_format & ~is_zero
        parsed = pd.to_datetime(txt.where(real_format), format="%Y-%m-%d", errors="coerce")
        cannot = s[real_format & parsed.isna()]
        good, stamp = s[parsed.notna()], parsed[parsed.notna()]
        before_1990 = int(good[stamp < pd.Timestamp("1990-01-01")].sum())
        after_crawl = int(good[stamp > pd.Timestamp("2018-05-25")].sum())
        print(f"--- {label}: {total:,} rows, {len(s):,} distinct texts")
        print(f"  empty {int(s[is_empty].sum()):,} | 0000-00-00 {int(s[is_zero].sum()):,} | other formats {int(other.sum()):,} {other.head(5).to_dict()}")
        print(f"  right format but not a real date (for example 2010-00-00 or 2018-02-31): {int(cannot.sum()):,} {cannot.head(5).to_dict()}")
        if len(good):
            by_md = good.groupby(good.index.str[5:10]).sum().sort_values(ascending=False)
            by_year = good.groupby(stamp.dt.year).sum().sort_values(ascending=False)
            share_jan1 = by_md.get("01-01", 0) / good.sum()
            print(f"  real dates: {int(good.sum()):,} | earliest {stamp.min().date()} | latest {stamp.max().date()}")
            print(f"  before 1990: {before_1990:,} | after the crawl (2018-05-25): {after_crawl:,}")
            print("  most frequent month-day:", by_md.head(5).to_dict(), f"| share of 01-01: {share_jan1:.2%} (evenly spread: 0.27%)")
            print("  most frequent years:", by_year.head(5).to_dict())

    summarize_dates(date_counts["my_start_date"], "my_start_date")
    print()
    summarize_dates(date_counts["my_finish_date"], "my_finish_date")
    print(f"\nfinish date before start date: {finish_before_start:,} of {both_real:,} rows that have two real dates")

    print("\nmy_last_updated (seconds since 1970):")
    print("  earliest:", pd.to_datetime(upd_min, unit="s"), "| latest:", pd.to_datetime(upd_max, unit="s"))
    print(f"  <= 0: {upd_nonpositive:,} | before November 2004 (MyAnimeList did not exist): {upd_before_site:,} | after 2018-05-25: {upd_after_crawl:,}")
else:
    print("Skipped (RUN_LISTS = False)")


In [ ]:
# B4. anime_id and usernames of the list table
if RUN_LISTS:
    orphan_ids = ids - known_ids
    print(f"anime_id in the list but not in AnimeList: {len(orphan_ids):,} ids, {orphan_rows:,} rows"
          f" | anime without any list row: {len(known_ids - ids):,} of {len(known_ids):,}")

    listed = pd.Series(sorted(names))
    low = listed.str.lower()
    clash = low.duplicated(keep=False)
    print(f"\nusernames in the list table: {len(listed):,} | empty username cells: {empty_username:,}")
    print("distinct after lowercase:", f"{low.nunique():,}", "| names that differ only by capital letters:", int(clash.sum()))
    print("  examples:", listed[clash].head(10).tolist())
    print("  with a space at the start or the end:", int((listed != listed.str.strip()).sum()))

    user_names = set(users["username"])
    lower_users = set(users["username"].str.lower())
    not_found = [n for n in listed if n not in user_names]
    only_case = [n for n in not_found if n.lower() in lower_users]
    print("list usernames not found in UserList (exact text):", len(not_found),
          "| found if capital letters are ignored:", len(only_case), "| examples:", not_found[:5])
else:
    print("Skipped (RUN_LISTS = False)")


In [ ]:
# B5. my_watched_episodes against the episode count of the anime
if RUN_LISTS:
    print(f"rows whose anime has a known episode count (> 0): {ep_known:,} ({ep_known / rows:.1%})")
    print(f"my_watched_episodes above the episode count of the anime: {watched_over:,}")
    print(f"completed rows: {completed_rows:,} | with my_watched_episodes == 0: {completed_zero:,}"
          f" | below the episode count: {completed_less:,}")
else:
    print("Skipped (RUN_LISTS = False)")


### Observations for part 7B

**TODO (fill in after the run):**
- B2: how many `my_tags` are placeholder-like, and which words? Do the most frequent tags look like real tags? How many contain an HTML code?
- B3: how many start and finish dates are `0000-00-00`, partial (`2010-00-00`), impossible, before 1990, or after the crawl? Is January 1 over-represented (compare with 0.27%)? How many finish dates come before the start date? Is `my_last_updated` ever before 2004 or after the crawl?
- B4: are there `anime_id` values that do not exist in `AnimeList`? Do usernames of the list table differ from those of `UserList` only by capital letters or spaces?
- B5: how many rows have more episodes watched than the anime has? How many completed rows have 0 or too few watched episodes?
- Which of these problems is new for the summary of step 3 (to add after the merge)?
